# Vuln2Action - Kaggle T5 smoke test

M?c ti?u: ki?m tra DDP 2x T4, fp16, ??c dataset 114.560 CVE, checkpoint v? resume
tr??c khi ch?y stage 1 full.

Dataset c?n c?: `minhiw/vuln2action-flow-114560`.
Kh?ng c? exploit source code trong dataset.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os

print("input roots:", os.listdir("/kaggle/input") if Path("/kaggle/input").exists() else [])
hits = sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits:
    raise FileNotFoundError("flow_train.jsonl not found under /kaggle/input")

FLOW_DIR = Path(hits[0]).parent
os.environ["FLOW_DIR"] = str(FLOW_DIR)
print("FLOW_DIR:", FLOW_DIR)
for name in ["flow_train.jsonl", "flow_validation.jsonl", "flow_stats.json"]:
    p = FLOW_DIR / name
    print(name, p.exists(), f"{p.stat().st_size / 1024 / 1024:.1f} MiB" if p.exists() else "")


In [ ]:
!python scripts/train_t5.py   --dataset "$FLOW_DIR"   --output-dir /kaggle/working/smoke   --model-name t5-small   --target-field target_techniques   --max-input-length 128   --max-target-length 128   --per-device-batch-size 4   --gradient-accumulation-steps 1   --max-steps 20   --save-steps 10   --eval-steps 10   --logging-steps 5   --limit 500   --fp16   --dry-run


In [ ]:
!python -m torch.distributed.run --nproc_per_node=2 scripts/train_t5.py   --dataset "$FLOW_DIR"   --output-dir /kaggle/working/smoke   --model-name t5-small   --target-field target_techniques   --max-input-length 128   --max-target-length 128   --per-device-batch-size 4   --gradient-accumulation-steps 1   --max-steps 20   --save-steps 10   --eval-steps 10   --logging-steps 5   --limit 500   --fp16


## N?u smoke test qua

Ch?y stage 1 T5-base, benchmark 1 epoch tr??c. Kh?ng ch?y 50 epoch ngay.

```bash
python -m torch.distributed.run --nproc_per_node=2 scripts/train_t5.py \
  --dataset /kaggle/input/vuln2action-flow-114560 \
  --output-dir /kaggle/working/t5_stage1 \
  --model-name t5-base \
  --target-field target_techniques \
  --max-input-length 256 \
  --max-target-length 256 \
  --per-device-batch-size 8 \
  --gradient-accumulation-steps 4 \
  --num-train-epochs 1 \
  --save-steps 500 \
  --eval-steps 500 \
  --fp16 --gradient-checkpointing --auto-resume
```

L?u checkpoint v?o Kaggle Dataset ho?c Drive sau m?i l?n ch?y.
